# Day 2 — PySpark Practice Questions
## Topic: upper · lower · initcap · trim · regexp_replace · split · substring · concat_ws

---

> **Rules:**
> - These are the **exact same 3 problems as SQL Day 2** — same data, same business logic
> - Solve using the **PySpark DataFrame API** (not SparkSQL strings)
> - Each question has concept warm-up cells — run them first
> - Each question has a setup cell that creates the DataFrame — run it before solving
> - Match the **exact expected output** (columns, values, order)

| # | Difficulty | Topic Focus |
|---|-----------|-------------|
| Q1 | 🟢 Easy | `lower()` + `trim()` + `initcap()` + `upper()` |
| Q2 | 🟡 Medium | `regexp_replace()` + `length()` + `instr()` + `when()` |
| Q3 | 🔴 Hard | `split()` + `substring()` + `concat_ws()` + multi-step cleaning |

---
## Setup — SparkSession (run once)

In [1]:
import os

os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("Environment set.")

Environment set.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit,
    upper, lower, initcap,
    trim, ltrim, rtrim,
    regexp_replace, split, substring,
    length, instr,
    concat, concat_ws,
    when, coalesce
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

spark = SparkSession.builder \
    .appName("Day02_PracticeQuestions") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready.")

Spark 3.5.6 ready.


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Day02Practice').master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

---
---

## Q1 — 🟢 Easy
### Standardise a Product Catalogue

---

### Concept Warm-Up — lower, upper, initcap, trim

In [7]:
data = [
    ("LAPTOP PRO 15", 393993),
    ("wireless MOUSE", 393993),
    ("MECHANICAL Keyboard", 39399)
]

In [8]:
demo = spark.createDataFrame(
    data, ['orginal', 'price']
)

In [9]:
demo.show()

+-------------------+------+
|            orginal| price|
+-------------------+------+
|      LAPTOP PRO 15|393993|
|     wireless MOUSE|393993|
|MECHANICAL Keyboard| 39399|
+-------------------+------+



In [10]:
data = [
    {"id": 1, "name": "Alice", "age": 25},
    {"id": 2, "name": "Bob", "age": 30}
]

In [14]:
df = spark.createDataFrame(data)

In [15]:
df.show()

+---+---+-----+
|age| id| name|
+---+---+-----+
| 25|  1|Alice|
| 30|  2|  Bob|
+---+---+-----+



In [ ]:
demo = spark.createDataFrame([
    ("LAPTOP PRO 15",),
    ("wireless MOUSE",),
    ("MECHANICAL Keyboard",),
], ["original"])


In [16]:
# Warm-up 1: lower / upper / initcap in action
demo = spark.createDataFrame([
    ("LAPTOP PRO 15",),
    ("wireless MOUSE",),
    ("MECHANICAL Keyboard",),
], ["original"])

demo.select(
    col("original"),
    lower(col("original")).alias("lower_result"),
    upper(col("original")).alias("upper_result"),
    initcap(col("original")).alias("initcap_result")
).show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|original           |lower_result       |upper_result       |initcap_result     |
+-------------------+-------------------+-------------------+-------------------+
|LAPTOP PRO 15      |laptop pro 15      |LAPTOP PRO 15      |Laptop Pro 15      |
|wireless MOUSE     |wireless mouse     |WIRELESS MOUSE     |Wireless Mouse     |
|MECHANICAL Keyboard|mechanical keyboard|MECHANICAL KEYBOARD|Mechanical Keyboard|
+-------------------+-------------------+-------------------+-------------------+



In [21]:
demo.select(
    col('original'),
    upper(col('original')).alias('upper_result'),
    lower(col("original")).alias("lower_result"),
    initcap(col("original")).alias("initcap_result")
).show()

+-------------------+-------------------+-------------------+-------------------+
|           original|       upper_result|       lower_result|     initcap_result|
+-------------------+-------------------+-------------------+-------------------+
|      LAPTOP PRO 15|      LAPTOP PRO 15|      laptop pro 15|      Laptop Pro 15|
|     wireless MOUSE|     WIRELESS MOUSE|     wireless mouse|     Wireless Mouse|
|MECHANICAL Keyboard|MECHANICAL KEYBOARD|mechanical keyboard|Mechanical Keyboard|
+-------------------+-------------------+-------------------+-------------------+



In [22]:
demo2 = spark.createDataFrame([
    ("  Laptop Pro 15  ",),
    ("  USB-C Hub  ",),
], ["original"])

In [23]:
demo2.show()

+-----------------+
|         original|
+-----------------+
|  Laptop Pro 15  |
|      USB-C Hub  |
+-----------------+



In [25]:
demo2.select(
    col('original'),
    trim(col('original')).alias('trimmed')
).show()

+-----------------+-------------+
|         original|      trimmed|
+-----------------+-------------+
|  Laptop Pro 15  |Laptop Pro 15|
|      USB-C Hub  |    USB-C Hub|
+-----------------+-------------+



In [26]:
demo2.select(
    col('original'),
    length(col('original')).alias('original_length'),
    length(trim(col('original'))).alias('trimmed_length')
).show()

+-----------------+---------------+--------------+
|         original|original_length|trimmed_length|
+-----------------+---------------+--------------+
|  Laptop Pro 15  |             17|            13|
|      USB-C Hub  |             13|             9|
+-----------------+---------------+--------------+



In [ ]:
# Warm-up 2: trim removes leading/trailing spaces
demo2 = spark.createDataFrame([
    ("  Laptop Pro 15  ",),
    ("  USB-C Hub  ",),
], ["original"])

demo2.select(
    col("original"),
    trim(col("original")).alias("trimmed"),
    length(col("original")).alias("original_length"),
    length(trim(col("original"))).alias("trimmed_length")
).show(truncate=False)

+--------------------+--------------------+---------------+----------------+
|original            |trimmed             |original_length|trimmed_length  |
+--------------------+--------------------+---------------+----------------+
|  Laptop Pro 15     |Laptop Pro 15       |18             |13              |
|  USB-C Hub         |USB-C Hub           |12             |9               |
+--------------------+--------------------+---------------+----------------+


In [ ]:
# Warm-up 3: combine trim + initcap — the standard product name cleaning chain
demo3 = spark.createDataFrame([
    ("  Laptop Pro 15  ",),
    ("wireless MOUSE",),
    ("MECHANICAL Keyboard",),
], ["raw"])

demo3.select(
    col("raw"),
    initcap(trim(col("raw"))).alias("name_clean")
).show(truncate=False)

+-------------------+--------------+
|raw                |name_clean    |
+-------------------+--------------+
|  Laptop Pro 15    |Laptop Pro 15 |
|wireless MOUSE     |Wireless Mouse|
|MECHANICAL Keyboard|Mechanical Keyboard|
+-------------------+--------------+


---

### Problem Statement

The product catalogue DataFrame has inconsistent formatting.  
Write a transformation that returns a **cleaned version** with:
1. `product_name` — trimmed and title-cased (`initcap(trim(...))`)
2. `category` — trimmed and uppercased
3. `sku` — trimmed and uppercased
4. `price` — unchanged

Sort by `category` ascending, then `product_name` ascending.

### DataFrame Schema

| Column | Type | Notes |
|--------|------|-------|
| product_id | IntegerType | |
| product_name | StringType | Mixed case, extra spaces |
| category | StringType | Random case |
| sku | StringType | Not consistently uppercase |
| price | DoubleType | Clean |

In [ ]:
# Q1 Setup
products_data = [
    (1,  '  Laptop Pro 15  ',            'ELECTRONICS', 'LP-15-2024',   85000.0),
    (2,  'wireless MOUSE',               'Electronics', 'WM-001-BLK',    1500.0),
    (3,  'MECHANICAL Keyboard',          'electronics', 'MK-TKL-2023',   4500.0),
    (4,  '  USB-C Hub   ',               'Accessories', 'UC-HUB-7P',     2200.0),
    (5,  'noise Cancelling headphones',  'ELECTRONICS', 'NCH-PRO-X',    12000.0),
    (6,  'Cotton T-Shirt  ',             'CLOTHING',    'CTS-M-WHT',      599.0),
    (7,  '  DENIM JEANS',                'Clothing',    'DJ-32-BLU',     1999.0),
    (8,  'running shoes  ',              'clothing',    'RS-42-BLK',     3499.0),
    (9,  'Python Programming',           'BOOKS',       'BK-PY-ADV',      799.0),
    (10, '  data engineering handbook',  'Books',       'BK-DE-2024',    1299.0),
    (11, 'Yoga Mat',                     'SPORTS',      'YM-6MM-PRP',    1199.0),
    (12, '  Dumbbell Set  ',             'sports',      'DB-10KG-SET',   3999.0),
]

products_schema = StructType([
    StructField("product_id",   IntegerType(), True),
    StructField("product_name", StringType(),  True),
    StructField("category",     StringType(),  True),
    StructField("sku",          StringType(),  True),
    StructField("price",        DoubleType(),  True),
])

df_products = spark.createDataFrame(products_data, schema=products_schema)
print("df_products created — 12 rows:")
df_products.show(3, truncate=False)

df_products created — 12 rows:
+----------+---------------------------+-----------+-----------+--------+
|product_id|product_name               |category   |sku        |price   |
+----------+---------------------------+-----------+-----------+--------+
|1         |  Laptop Pro 15            |ELECTRONICS|LP-15-2024 |85000.0 |
|2         |wireless MOUSE             |Electronics|WM-001-BLK |1500.0  |
|3         |MECHANICAL Keyboard        |electronics|MK-TKL-2023|4500.0  |
+----------+---------------------------+-----------+-----------+--------+
only showing top 3 rows


### Expected Output

*(12 rows, sorted by category ASC, then product_name ASC)*

```
+---------------------------+-----------+-----------+--------+
|product_name               |category   |sku        |price   |
+---------------------------+-----------+-----------+--------+
|Usb-C Hub                  |ACCESSORIES|UC-HUB-7P  |2200.0  |
|Data Engineering Handbook  |BOOKS      |BK-DE-2024 |1299.0  |
|Python Programming         |BOOKS      |BK-PY-ADV  |799.0   |
|Cotton T-Shirt             |CLOTHING   |CTS-M-WHT  |599.0   |
|Denim Jeans                |CLOTHING   |DJ-32-BLU  |1999.0  |
|Running Shoes              |CLOTHING   |RS-42-BLK  |3499.0  |
|Laptop Pro 15              |ELECTRONICS|LP-15-2024 |85000.0 |
|Mechanical Keyboard        |ELECTRONICS|MK-TKL-2023|4500.0  |
|Noise Cancelling Headphones|ELECTRONICS|NCH-PRO-X  |12000.0 |
|Wireless Mouse             |ELECTRONICS|WM-001-BLK |1500.0  |
|Dumbbell Set               |SPORTS     |DB-10KG-SET|3999.0  |
|Yoga Mat                   |SPORTS     |YM-6MM-PRP |1199.0  |
+---------------------------+-----------+-----------+--------+
```

**PySpark API hints:**
- `initcap(trim(col("product_name")))` for product_name
- `upper(trim(col("category")))` for category and sku
- `.orderBy("category", "product_name")` — operates on the cleaned columns, not raw
- Use `.select()` or `.withColumn()` to build the cleaned DataFrame

In [ ]:
# Q1 — Write your solution here
# Use: initcap(), trim(), upper(), orderBy()


---
---

## Q2 — 🟡 Medium
### Validate and Flag User Contact Information

---

### Concept Warm-Up — regexp_replace, length, instr, when

In [36]:
# Warm-up 1: regexp_replace — strip all non-digit characters from phone
phones_demo = spark.createDataFrame([
    ("+91-9876543210",),
    ("(098) 761-00003",),
    ("987.610.0004",),
    ("abcd-efgh",),
], ["phone_raw"])

phones_demo.select(
    col("phone_raw"),
    regexp_replace(col("phone_raw"), r"[^0-9]", "").alias("digits_only")
).show(truncate=False)


string1 = "\n"
print(string1)
string2 = r"\n"
print(string2)
print("Data@Engineering")

# \n	New line
# \t	Horizontal tab
# \r	Carriage return
# \b	Backspace
# \f	Form feed
# \v	Vertical tab
# \\	Backslash
# \'	Single quote
# \"	Double quote
# \a	Bell/alert sound
# \0	Null character
# \ooo	Character with octal value

+---------------+------------+
|phone_raw      |digits_only |
+---------------+------------+
|+91-9876543210 |919876543210|
|(098) 761-00003|09876100003 |
|987.610.0004   |9876100004  |
|abcd-efgh      |            |
+---------------+------------+



\n
Data@Engineering


In [43]:
phones_demo2 = spark.createDataFrame([
    ("+91-9876543210",),
    ("9876543211",),
    ("abcd-efgh",),
    ("987654321",),
], ["phone_raw"])

phones_demo2 = phones_demo2.withColumn(
    "digits_only", regexp_replace(col("phone_raw"), r"[^0-9]", "")
)

In [44]:
phones_demo2.show()

+--------------+------------+
|     phone_raw| digits_only|
+--------------+------------+
|+91-9876543210|919876543210|
|    9876543211|  9876543211|
|     abcd-efgh|            |
|     987654321|   987654321|
+--------------+------------+



In [48]:
# Warm-up 2: length on cleaned phone — valid if exactly 10 digits
phones_demo2 = spark.createDataFrame([
    ("+91-9876543210",),
    ("9876543211",),
    ("abcd-efgh",),
    ("987654321",),
], ["phone_raw"])

phones_demo2.withColumn(
    "digits_only", regexp_replace(col("phone_raw"), r"[^0-9]", "")
).select(
    col("phone_raw"),
    col("digits_only"),
    length(col("digits_only")).alias("digit_count"),
    when(length(col("digits_only")) == 10, lit("valid"))
    .otherwise(concat(lit("invalid ("), length(col("digits_only")), lit(" digits)")))
    .alias("phone_status")
).show(truncate=False)

+--------------+------------+-----------+-------------------+
|phone_raw     |digits_only |digit_count|phone_status       |
+--------------+------------+-----------+-------------------+
|+91-9876543210|919876543210|12         |invalid (12 digits)|
|9876543211    |9876543211  |10         |valid              |
|abcd-efgh     |            |0          |invalid (0 digits) |
|987654321     |987654321   |9          |invalid (9 digits) |
+--------------+------------+-----------+-------------------+



In [52]:
from pyspark.sql.functions import expr
# Warm-up 3: instr to validate email has '@' AND '.' after '@'
emails_demo = spark.createDataFrame([
    ("alice@gmail.com",),
    ("bob AT yahoo DOT com",),
    ("jack@",),
    ("david@gmail",),
], ["email"])

emails_demo.withColumn("at_pos", instr(col("email"), "@")) \
           .withColumn("domain_part", expr("substring(email, instr(email, '@') + 1, 100)")) \
           .select(
               col("email"),
               col("at_pos"),
               (instr(col("email"), ".") > instr(col("email"), "@")).alias("has_dot"),
               when(
                   (instr(col("email"), "@") > 0) &
                   (instr(col("email"), ".") > instr(col("email"), "@")),
                   lit("valid")
               ).otherwise(lit("invalid")).alias("email_status")
           ).show(truncate=False)

+--------------------+------+-------+------------+
|email               |at_pos|has_dot|email_status|
+--------------------+------+-------+------------+
|alice@gmail.com     |6     |true   |valid       |
|bob AT yahoo DOT com|0     |false  |invalid     |
|jack@               |5     |false  |invalid     |
|david@gmail         |6     |false  |invalid     |
+--------------------+------+-------+------------+



---

### Problem Statement

Same as SQL Q2: validate user contact info and produce a DQ report.  
For each **active** user:

1. **Phone validity:** After `regexp_replace` to keep digits only, phone must be exactly **10 digits**
2. **Email validity:** Must contain `'@'` AND have at least one `'.'` after the `'@'`

Return:
- `user_id`, `username`
- `phone_clean` — digits only
- `phone_status` — `'valid'` or `'invalid (N digits)'`
- `email_status` — `'valid'` or `'invalid'`
- `overall_status` — `'clean'` if both valid, otherwise `'needs fix'`

Include only **active** users. Sort by `user_id`.

### DataFrame Schema

| Column | Type |
|--------|------|
| user_id | IntegerType |
| username | StringType |
| email | StringType |
| phone | StringType |
| country_code | StringType |
| account_status | StringType |

In [ ]:
# Q2 Setup
users_data = [
    (1,  'alice_j',  'alice@gmail.com',       '+91-9876543210',   'IN', 'active'),
    (2,  'bob_s',    'bob AT yahoo DOT com',  '9876543211',       'IN', 'active'),
    (3,  'carol_w',  'carol@hotmail.com',     'abcd-efgh',        'US', 'active'),
    (4,  'dave_b',   'david@gmail',           '+1-5559876543',    'US', 'active'),
    (5,  'eve_n',    'eve@gmail.com',         '9876543214',       'IN', 'inactive'),
    (6,  'frank_s',  'frank@outlook.com',     '(+44) 7911123456', 'UK', 'active'),
    (7,  'grace_p',  'GRACE yahoo com',       '9876543216',       'IN', 'active'),
    (8,  'henry_d',  'henry@gmail.com',       '987654321',        'IN', 'active'),
    (9,  'irene_v',  'irene@gmail.com',       '+91-9876543218',   'IN', 'active'),
    (10, 'jack_k',   'jack@',                 '9876543219',       'IN', 'active'),
]

users_schema = StructType([
    StructField("user_id",        IntegerType(), True),
    StructField("username",       StringType(),  True),
    StructField("email",          StringType(),  True),
    StructField("phone",          StringType(),  True),
    StructField("country_code",   StringType(),  True),
    StructField("account_status", StringType(),  True),
])

df_users = spark.createDataFrame(users_data, schema=users_schema)
print("df_users created — 10 rows:")
df_users.show(3, truncate=False)

df_users created — 10 rows:
+-------+---------+--------------------+----------------+------------+--------------+
|user_id|username |email               |phone           |country_code|account_status|
+-------+---------+--------------------+----------------+------------+--------------+
|1      |alice_j  |alice@gmail.com     |+91-9876543210  |IN          |active        |
|2      |bob_s    |bob AT yahoo DOT com|9876543211      |IN          |active        |
|3      |carol_w  |carol@hotmail.com   |abcd-efgh       |US          |active        |
+-------+---------+--------------------+----------------+------------+--------------+
only showing top 3 rows


### Expected Output

*(9 rows — user_id 5 is inactive, excluded. Sorted by user_id.)*

```
+-------+---------+------------+--------------------+------------+--------------+
|user_id|username |phone_clean |phone_status        |email_status|overall_status|
+-------+---------+------------+--------------------+------------+--------------+
|1      |alice_j  |919876543210|invalid (12 digits) |valid       |needs fix     |
|2      |bob_s    |9876543211  |valid               |invalid     |needs fix     |
|3      |carol_w  |            |invalid (0 digits)  |valid       |needs fix     |
|4      |dave_b   |15559876543 |invalid (11 digits) |invalid     |needs fix     |
|6      |frank_s  |447911123456|invalid (12 digits) |valid       |needs fix     |
|7      |grace_p  |9876543216  |valid               |invalid     |needs fix     |
|8      |henry_d  |987654321   |invalid (9 digits)  |valid       |needs fix     |
|9      |irene_v  |919876543218|invalid (12 digits) |valid       |needs fix     |
|10     |jack_k   |9876543219  |valid               |invalid     |needs fix     |
+-------+---------+------------+--------------------+------------+--------------+
```

**PySpark API hints:**
- `phone_clean = regexp_replace(col("phone"), r"[^0-9]", "")`
- `phone_status`: `when(length(phone_clean) == 10, "valid").otherwise(concat(lit("invalid ("), length(phone_clean), lit(" digits)")))`
- `email_status`: `when((instr(col("email"),"@") > 0) & (instr(col("email"),".") > instr(col("email"),"@")), "valid").otherwise("invalid")`
- `overall_status`: `when((phone_status_col == "valid") & (email_status_col == "valid"), "clean").otherwise("needs fix")`
- Use `.withColumn()` to build each computed column step by step before the final `.select()`

In [ ]:
# Q2 — Write your solution here
# Use: filter(), regexp_replace(), length(), instr(), when(), concat(), lit()


---
---

## Q3 — 🔴 Hard
### Parse and Clean a Pipe-Delimited Transaction Log

---

### Concept Warm-Up — split, substring, concat_ws

In [ ]:
# Warm-up 1: split on pipe — extract positional fields (0-indexed!)
demo = spark.createDataFrame([("a|b|c|d|e",)], ["log_line"])
demo.select(
    col("log_line"),
    split(col("log_line"), r"\|")[0].alias("field1"),   # 'a' — index 0
    split(col("log_line"), r"\|")[1].alias("field2"),   # 'b' — index 1
    split(col("log_line"), r"\|")[2].alias("field3"),
    split(col("log_line"), r"\|")[3].alias("field4"),
    split(col("log_line"), r"\|")[4].alias("field5"),
).show(truncate=False)

+----------+------+--------+------+-------+-------+
|log_line  |field1|field2  |field3|field4 |field5 |
+----------+------+--------+------+-------+-------+
|a|b|c|d|e|a     |b       |c     |d      |e      |
+----------+------+--------+------+-------+-------+


In [ ]:
# Warm-up 2: substring to extract date parts (1-based position)
demo2 = spark.createDataFrame([("2024-01-15",)], ["full_date"])
demo2.select(
    col("full_date"),
    substring(col("full_date"), 1, 4).alias("year"),
    substring(col("full_date"), 6, 2).alias("month"),
    substring(col("full_date"), 9, 2).alias("day"),
    substring(col("full_date"), 1, 7).alias("ym"),
).show(truncate=False)

+----------+----+-----+---+-------+
|full_date |year|month|day|ym     |
+----------+----+-----+---+-------+
|2024-01-15|2024|01   |15 |2024-01|
+----------+----+-----+---+-------+


In [ ]:
# Warm-up 3: concat_ws — separator-joined, skips NULLs
demo3 = spark.createDataFrame([("TXN-001", "Alice", "5000", "OK")], ["a", "b", "c", "d"])
demo3.select(
    concat_ws(" | ", col("a"), col("b"), col("c"), col("d")).alias("all_present"),
    concat_ws(" | ", lit(None), col("b"), col("c"), col("d")).alias("with_null")  # skips NULL
).show(truncate=False)

+----------------------------+--------------------+
|all_present                 |with_null           |
+----------------------------+--------------------+
|TXN-001 | Alice | 5000 | OK|Alice | 5000 | OK   |
+----------------------------+--------------------+


In [ ]:
# Warm-up 4: trim + initcap after split — raw fields have spaces
demo4 = spark.createDataFrame([
    ("  Alice Johnson  ",),
    ("GRACE PATEL",),
], ["raw_field"])
demo4.select(
    col("raw_field"),
    initcap(trim(col("raw_field"))).alias("cleaned")
).show(truncate=False)

+------------------------------------+------------------+
|raw_field                           |cleaned           |
+------------------------------------+------------------+
|  Alice Johnson                     |Alice Johnson     |
|GRACE PATEL                         |Grace Patel       |
+------------------------------------+------------------+


---

### Problem Statement

Same as SQL Q3: parse pipe-delimited log lines into structured, cleaned columns.  
Log format: `date|txn_id|txn_type|customer_name|amount|status`

Return (only **SUCCESS** transactions):
1. `txn_date` — field 1, as-is
2. `txn_id` — field 2, trimmed, uppercased
3. `txn_type` — field 3, trimmed, uppercased
4. `customer_name` — field 4, trimmed, initcap
5. `amount` — field 5, trimmed, cast to `DoubleType`
6. `status` — field 6, trimmed, uppercased
7. `txn_month` — YYYY-MM from txn_date (use `substring`)
8. `row_summary` — `txn_id | customer_name | amount | status` (use `concat_ws`)

Sort by `txn_date` ascending, then `txn_id` ascending.

### DataFrame Schema

| Column | Type | Notes |
|--------|------|-------|
| log_id | IntegerType | |
| log_line | StringType | Pipe-delimited, messy fields |

In [ ]:
# Q3 Setup
logs_data = [
    (1,  '2024-01-15|TXN-001|CREDIT|  Alice Johnson  |5000.00|SUCCESS'),
    (2,  '2024-01-15|TXN-002|DEBIT |bob smith        |1200.50|SUCCESS'),
    (3,  '2024-01-16|TXN-003|CREDIT|  CAROL WHITE    |8750.00|FAILED '),
    (4,  '2024-01-16|TXN-004|DEBIT |David Brown      |300.00 |SUCCESS'),
    (5,  '2024-01-17|TXN-005|CREDIT|eve nair         |  2500.00|PENDING'),
    (6,  '2024-01-17|TXN-006|DEBIT |  Frank Singh    |650.75 |SUCCESS'),
    (7,  '2024-01-18|TXN-007|CREDIT|GRACE PATEL      |11000.00|SUCCESS'),
    (8,  '2024-01-18|TXN-008|DEBIT |henry das        |450.00 |FAILED '),
    (9,  '2024-01-19|TXN-009|CREDIT|  Irene Verma    |3300.00|SUCCESS'),
    (10, '2024-01-19|TXN-010|DEBIT |jack KUMAR       |900.25 |PENDING'),
]

logs_schema = StructType([
    StructField("log_id",   IntegerType(), True),
    StructField("log_line", StringType(),  True),
])

df_logs = spark.createDataFrame(logs_data, schema=logs_schema)
print("df_logs created — 10 rows:")
df_logs.show(3, truncate=False)

df_logs created — 10 rows:
+------+-------------------------------------------------------+
|log_id|log_line                                               |
+------+-------------------------------------------------------+
|1     |2024-01-15|TXN-001|CREDIT|  Alice Johnson  |5000.00|SUCCESS|
|2     |2024-01-15|TXN-002|DEBIT |bob smith        |1200.50|SUCCESS|
|3     |2024-01-16|TXN-003|CREDIT|  CAROL WHITE    |8750.00|FAILED |
+------+-------------------------------------------------------+
only showing top 3 rows


### Expected Output

*(6 rows — SUCCESS transactions only, sorted by txn_date then txn_id)*

```
+----------+-------+--------+-------------+-------+-------+---------+---------------------------------------+
|txn_date  |txn_id |txn_type|customer_name|amount |status |txn_month|row_summary                            |
+----------+-------+--------+-------------+-------+-------+---------+---------------------------------------+
|2024-01-15|TXN-001|CREDIT  |Alice Johnson |5000.0 |SUCCESS|2024-01  |TXN-001 | Alice Johnson | 5000.0 | SUCCESS|
|2024-01-15|TXN-002|DEBIT   |Bob Smith     |1200.5 |SUCCESS|2024-01  |TXN-002 | Bob Smith | 1200.5 | SUCCESS    |
|2024-01-16|TXN-004|DEBIT   |David Brown   |300.0  |SUCCESS|2024-01  |TXN-004 | David Brown | 300.0 | SUCCESS  |
|2024-01-17|TXN-006|DEBIT   |Frank Singh   |650.75 |SUCCESS|2024-01  |TXN-006 | Frank Singh | 650.75 | SUCCESS  |
|2024-01-18|TXN-007|CREDIT  |Grace Patel   |11000.0|SUCCESS|2024-01  |TXN-007 | Grace Patel | 11000.0 | SUCCESS |
|2024-01-19|TXN-009|CREDIT  |Irene Verma   |3300.0 |SUCCESS|2024-01  |TXN-009 | Irene Verma | 3300.0 | SUCCESS  |
+----------+-------+--------+-------------+-------+-------+---------+---------------------------------------+
```

**PySpark API hints:**
- `fields = split(col("log_line"), r"\|")` — split once, reuse
- Access fields: `fields[0]` = date, `fields[1]` = txn_id, ..., `fields[5]` = status (0-indexed)
- Filter: `trim(upper(fields[5])) == "SUCCESS"`
- Cast amount: `trim(fields[4]).cast(DoubleType())`
- `txn_month`: `substring(fields[0], 1, 7)` — first 7 chars = YYYY-MM
- `row_summary`: `concat_ws(" | ", txn_id_col, customer_col, amount_col.cast(StringType()), status_col)`
- Tip: use `.withColumn()` to extract each field first, then `.select()` the final columns

In [ ]:
# Q3 — Write your solution here
# Use: split(), trim(), upper(), initcap(), substring(), concat_ws(), cast(), filter(), orderBy()


In [ ]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
